# Phase 6 - Abstractive Legal Summarization
Datasets to attach: (1) your Phase 1 checkpoints dataset (A_decoder.pt, B_encdec.pt, C_hybrid.pt, D_efficient.pt, bpe32k.json), (2) optional Phase_6_Summ train.json/test.json. GPU on, Internet on.
Experiments: E1 LoRA (decoder, also KD teacher), E2 Houlsby adapters (enc-dec), E3 LoRMA (hybrid), E4 Knowledge distillation (teacher A -> student D).

In [1]:
import os, sys, re, json, glob, math, time, random, subprocess, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
subprocess.run([sys.executable,"-m","pip","install","-q","tokenizers","rouge-score","bert-score","pandas"],check=False)
from tokenizers import Tokenizer
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda"
WORK="/kaggle/working"; OUT=f"{WORK}/p6"; os.makedirs(OUT,exist_ok=True)
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
inp = glob.glob("/kaggle/input/**/*.*", recursive=True)
def pick(suffix, must=None):
    c=[p for p in inp if p.endswith(suffix) and (must is None or must.lower() in p.lower())]
    return c[0] if c else None
def ckpt_path(name):
    c=[p for p in inp if p.endswith(f"{name}.pt")]
    assert c, f"{name}.pt not found in /kaggle/input"
    return c[0]

TOK_PATH = pick("bpe32k.json")
TRAIN = pick("train.json","summ") or pick("train.json")
TEST  = pick("test.json","summ")  or pick("test.json")
if not (TRAIN and TEST):
    from huggingface_hub import snapshot_download
    from kaggle_secrets import UserSecretsClient
    d = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                          allow_patterns=["Phase_6_Summ/*"], local_dir="/tmp/hf",
                          token=UserSecretsClient().get_secret("HF_TOKEN"))
    TRAIN=f"{d}/Phase_6_Summ/Train/train.json"; TEST=f"{d}/Phase_6_Summ/Test/test.json"
print(TOK_PATH, TRAIN, TEST, sep="\n")

def norm(x):
    if isinstance(x, dict):
        return [dict(id=k, **(v if isinstance(v, dict) else {"document": v})) for k, v in x.items()]
    return x
tr = norm(json.load(open(TRAIN, encoding="utf-8")))
te = norm(json.load(open(TEST,  encoding="utf-8")))
print(len(tr), len(te), list(tr[0].keys()))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/kaggle/input/datasets/samswe/bpefile/bpe32k.json
/tmp/hf/Phase_6_Summ/Train/train.json
/tmp/hf/Phase_6_Summ/Test/test.json
7030 100 ['id', 'document', 'summary']


In [3]:
tok = Tokenizer.from_file(TOK_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
VOCAB_SIZE = 32000
DOC_TOK, SUM_TOK = 1280, 256

def ht(ids, L):
    if len(ids) <= L: return ids
    h = int(L*0.7); return ids[:h] + ids[-(L-h):]
def cut_text(t):
    return t if len(t) <= 40000 else t[:28000] + " " + t[-12000:]
def enc_docs(texts, bs=256):
    out=[]
    for i in range(0, len(texts), bs):
        out += [ht(e.ids, DOC_TOK) for e in tok.encode_batch([cut_text(t) for t in texts[i:i+bs]])]
    return out

doc_ids  = enc_docs([d["document"] for d in tr])
sum_txt  = [d["summary"] for d in tr]
sum_ids  = [e.ids[:SUM_TOK-1] + [EOS_ID] for e in tok.encode_batch(sum_txt)]
test_ids = enc_docs([d["document"] for d in te])

sl = np.array([len(s) for s in sum_ids])
print("summary tokens mean/p90/max:", sl.mean().round(1), np.percentile(sl,90), sl.max())
print("docs >= DOC_TOK:", np.mean([len(d)>=DOC_TOK for d in doc_ids]).round(3))

rng = np.random.RandomState(SEED); perm = rng.permutation(len(tr))
val_idx, train_idx = list(perm[:200]), list(perm[200:])

def batches(idxs, bs, shuffle=True):
    order = sorted(idxs, key=lambda i: len(doc_ids[i]))
    while True:
        chunks = [order[i:i+bs] for i in range(0, len(order)-bs+1, bs)]
        if shuffle: random.shuffle(chunks)
        for c in chunks: yield c

summary tokens mean/p90/max: 253.3 256.0 256
docs >= DOC_TOK: 0.943


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rotary_freqs(dim, max_seq, base=10000.0, device="cpu"):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    f = torch.outer(torch.arange(max_seq, device=device).float(), inv)
    return torch.cos(f), torch.sin(f)

def apply_rotary(x, cos, sin):
    d = x.shape[-1]; x1, x2 = x[..., :d//2], x[..., d//2:]
    cos = cos[:x.shape[-2], :d//2][None, None]; sin = sin[:x.shape[-2], :d//2][None, None]
    return torch.cat([x1*cos - x2*sin, x2*cos + x1*sin], -1)

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False); self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x): return self.w3(F.silu(self.w1(x)) * self.w2(x))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.n_heads, self.hd = n_heads, dim // n_heads
        self.qkv = nn.Linear(dim, 3*dim, bias=False); self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rotary_freqs(self.hd, max_seq)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.hd).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).reshape(B, T, C))

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq, mlp_mult=4):
        super().__init__()
        self.norm1 = RMSNorm(dim); self.attn = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim); self.mlp = SwiGLU(dim, int(dim*mlp_mult*2/3))
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); return x + self.mlp(self.norm2(x))

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_layers=12, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([DecoderBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight; self.max_seq = max_seq
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for b in self.blocks: x = b(x)
        return self.head(self.norm_f(x)), None

class Attn(nn.Module):
    def __init__(self, dim, n_heads, max_seq, n_kv_heads=None, causal=True, window=None, rope=True):
        super().__init__()
        self.h, self.kv = n_heads, (n_kv_heads or n_heads); self.hd = dim // n_heads
        self.causal, self.window, self.rope = causal, window, rope
        self.q = nn.Linear(dim, self.h*self.hd, bias=False)
        self.k = nn.Linear(dim, self.kv*self.hd, bias=False)
        self.v = nn.Linear(dim, self.kv*self.hd, bias=False)
        self.proj = nn.Linear(self.h*self.hd, dim, bias=False)
        if rope:
            cos, sin = rotary_freqs(self.hd, max_seq)
            self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def _mask(self, T, device):
        i = torch.arange(T, device=device)[:, None]; j = torch.arange(T, device=device)[None, :]
        return (j <= i) & (i - j < self.window)
    def forward(self, x, ctx=None):
        B, T, C = x.shape; src = x if ctx is None else ctx; S = src.size(1)
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        v = self.v(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        if self.rope and ctx is None:
            q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        if self.kv != self.h:
            r = self.h // self.kv; k, v = k.repeat_interleave(r, 1), v.repeat_interleave(r, 1)
        if ctx is None and self.causal and self.window:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=self._mask(T, x.device))
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=(self.causal and ctx is None))
        return self.proj(out.transpose(1, 2).reshape(B, T, self.h*self.hd))

class EncBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.attn = RMSNorm(dim), Attn(dim, nh, ms, causal=False)
        self.n2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim*mlp_mult*2/3))
    def forward(self, x):
        x = x + self.attn(self.n1(x)); return x + self.mlp(self.n2(x))

class CrossDecBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.self_attn = RMSNorm(dim), Attn(dim, nh, ms, causal=True)
        self.n2, self.cross = RMSNorm(dim), Attn(dim, nh, ms, causal=False, rope=False)
        self.n3, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim*mlp_mult*2/3))
    def forward(self, x, enc):
        x = x + self.self_attn(self.n1(x)); x = x + self.cross(self.n2(x), ctx=enc)
        return x + self.mlp(self.n3(x))

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_enc=6, n_dec=6, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.enc = nn.ModuleList([EncBlock(dim, n_heads, max_seq) for _ in range(n_enc)])
        self.dec = nn.ModuleList([CrossDecBlock(dim, n_heads, max_seq) for _ in range(n_dec)])
        self.norm_e, self.norm_f = RMSNorm(dim), RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False); self.head.weight = self.tok_emb.weight
        self.max_seq = max_seq
    def encode(self, src):
        e = self.tok_emb(src)
        for b in self.enc: e = b(e)
        return self.norm_e(e)
    def decode(self, dec_in, enc_out):
        x = self.tok_emb(dec_in)
        for b in self.dec: x = b(x, enc_out)
        return self.head(self.norm_f(x))

class GatedLinearRecurrence(nn.Module):
    def __init__(self, dim, chunk=64):
        super().__init__()
        self.inp = nn.Linear(dim, 4*dim, bias=False)
        self.conv = nn.Conv1d(dim, dim, 4, groups=dim, padding=3)
        self.out = nn.Linear(dim, dim, bias=False); self.chunk = chunk
    def _scan(self, log_a, z):
        B, T, C = z.shape; L = self.chunk; pad = (-T) % L
        if pad: log_a, z = F.pad(log_a, (0,0,0,pad)), F.pad(z, (0,0,0,pad))
        n = (T + pad) // L
        log_a, z = log_a.reshape(B, n, L, C), z.reshape(B, n, L, C)
        A = log_a.cumsum(2); local = torch.exp(A) * torch.cumsum(torch.exp(-A) * z, 2)
        prev, outs = z.new_zeros(B, C), []
        for j in range(n):
            o = local[:, j] + torch.exp(A[:, j]) * prev[:, None]; prev = o[:, -1]; outs.append(o)
        return torch.stack(outs, 1).reshape(B, n*L, C)[:, :T]
    def forward(self, x):
        B, T, C = x.shape
        u, ga, gi, go = self.inp(x).chunk(4, -1)
        u = self.conv(u.transpose(1, 2))[..., :T].transpose(1, 2)
        log_a = -0.05 * torch.sigmoid(ga.float()); z = (torch.sigmoid(gi) * F.silu(u)).float()
        return self.out(self._scan(log_a, z).to(x.dtype) * F.silu(go))

class Block(nn.Module):
    def __init__(self, dim, mixer, mlp_mult=4):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim*mlp_mult*2/3))
    def forward(self, x):
        x = x + self.mixer(self.norm1(x)); return x + self.mlp(self.norm2(x))

class GenericLM(nn.Module):
    def __init__(self, make_block, vocab=VOCAB_SIZE, dim=768, n_layers=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([make_block(i) for i in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight; self.max_seq = max_seq
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for b in self.blocks: x = b(x)
        return self.head(self.norm_f(x)), None

def make_hybrid(dim=768, n_heads=12, max_seq=2048, attn_every=3):
    def f(i):
        return Block(dim, Attn(dim, n_heads, max_seq) if (i+1) % attn_every == 0 else GatedLinearRecurrence(dim))
    return f
def make_efficient(dim=768, n_heads=12, n_kv=4, window=256, global_every=4, max_seq=2048):
    def f(i):
        g = (i+1) % global_every == 0
        return Block(dim, Attn(dim, n_heads, max_seq, n_kv_heads=n_kv, window=None if g else window), mlp_mult=3)
    return f

def build(name):
    if name == "A_decoder":   return DecoderOnlyLM(dim=768, n_layers=12, n_heads=12)
    if name == "B_encdec":    return EncoderDecoderLM(dim=768, n_enc=6, n_dec=6, n_heads=12)
    if name == "C_hybrid":    return GenericLM(make_hybrid(), dim=768, n_layers=12)
    if name == "D_efficient": return GenericLM(make_efficient(), dim=768, n_layers=12)
def count_params(m): return sum(p.numel() for p in m.parameters())

In [5]:
COS, SIN = rotary_freqs(64, 2048, device=DEVICE)
def rope(x, pos0):
    T, d = x.shape[-2], x.shape[-1]
    c = COS[pos0:pos0+T, :d//2][None, None]; s = SIN[pos0:pos0+T, :d//2][None, None]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    return torch.cat([x1*c - x2*s, x2*c + x1*s], -1)
def sdpa(q, k, v, mask=None, causal=False):
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal)

def attn_step(a, h, cache, pos0):
    B, T, _ = h.shape
    q = a.q(h).view(B, T, a.h, a.hd).transpose(1, 2)
    k = a.k(h).view(B, T, a.kv, a.hd).transpose(1, 2)
    v = a.v(h).view(B, T, a.kv, a.hd).transpose(1, 2)
    q, k = rope(q, pos0).to(v.dtype), rope(k, pos0).to(v.dtype)
    if cache is not None: k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    if a.window and T == 1: k, v = k[:, :, -a.window:], v[:, :, -a.window:]; new = (k, v)
    elif a.window: new = (k[:, :, -a.window:], v[:, :, -a.window:])
    else: new = (k, v)
    if a.kv != a.h:
        r = a.h // a.kv; k, v = k.repeat_interleave(r, 1), v.repeat_interleave(r, 1)
    if T > 1: out = sdpa(q, k, v, mask=a._mask(T, h.device)) if a.window else sdpa(q, k, v, causal=True)
    else: out = sdpa(q, k, v)
    return a.proj(out.transpose(1, 2).reshape(B, T, a.h*a.hd)), new

def a_attn_step(a, h, cache, pos0):
    B, T, C = h.shape
    qkv = a.qkv(h).view(B, T, 3, a.n_heads, a.hd).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    q, k = rope(q, pos0).to(v.dtype), rope(k, pos0).to(v.dtype)
    if cache is not None: k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
    out = sdpa(q, k, v, causal=(T > 1))
    return a.proj(out.transpose(1, 2).reshape(B, T, C)), (k, v)

def A_forward(m, x, states, pos0):
    x = m.tok_emb(x); new = []
    for i, b in enumerate(m.blocks):
        o, c = a_attn_step(b.attn, b.norm1(x), None if states is None else states[i], pos0)
        x = x + o; x = x + b.mlp(b.norm2(x)); new.append(c)
    return m.head(m.norm_f(x[:, -1:]))[:, -1].float(), new

def glr_step(g, h, st):
    B, T, C = h.shape
    u, ga, gi, go = g.inp(h).chunk(4, -1); ut = u.transpose(1, 2)
    log_a = -0.05 * torch.sigmoid(ga.float())
    if st is None:
        uc = g.conv(ut)[..., :T].transpose(1, 2)
        z = (torch.sigmoid(gi) * F.silu(uc)).float(); hf = g._scan(log_a, z)
        new = (hf[:, -1].contiguous(), ut[..., -3:].float().contiguous()); hs = hf.to(h.dtype)
    else:
        hstate, cstate = st; win = torch.cat([cstate, ut.float()], 2)
        uc = ((win * g.conv.weight.squeeze(1).float()[None]).sum(-1) + g.conv.bias.float())[:, None, :]
        z = (torch.sigmoid(gi.float()) * F.silu(uc)).float()
        hn = torch.exp(log_a[:, 0]) * hstate + z[:, 0]
        new = (hn, win[..., 1:].contiguous()); hs = hn[:, None].to(h.dtype)
    return g.out(hs * F.silu(go)), new

def G_forward(m, x, states, pos0):
    x = m.tok_emb(x); new = []
    for i, b in enumerate(m.blocks):
        h, st = b.norm1(x), (None if states is None else states[i])
        o, c = attn_step(b.mixer, h, st, pos0) if isinstance(b.mixer, Attn) else glr_step(b.mixer, h, st)
        x = x + o; x = x + b.mlp(b.norm2(x)); new.append(c)
    return m.head(m.norm_f(x[:, -1:]))[:, -1].float(), new

def B_step(m, tok, st):
    x = m.tok_emb(tok[:, None]); B = x.size(0); new = []
    for i, b in enumerate(m.dec):
        o, c = attn_step(b.self_attn, b.n1(x), st["self"][i], st["pos"]); x = x + o; new.append(c)
        a = b.cross; q = a.q(b.n2(x)).view(B, 1, a.h, a.hd).transpose(1, 2); k, v = st["kv"][i]
        x = x + a.proj(sdpa(q, k, v).transpose(1, 2).reshape(B, 1, a.h*a.hd))
        x = x + b.mlp(b.n3(x))
    return m.head(m.norm_f(x))[:, -1].float(), dict(kv=st["kv"], self=new, pos=st["pos"] + 1)

def B_prefill(m, ids):
    enc = m.encode(ids); Bz, S, _ = enc.shape; kv = []
    for b in m.dec:
        a = b.cross
        kv.append((a.k(enc).view(Bz, S, a.kv, a.hd).transpose(1, 2), a.v(enc).view(Bz, S, a.kv, a.hd).transpose(1, 2)))
    bos = torch.full((Bz,), BOS_ID, device=ids.device)
    return B_step(m, bos, dict(kv=kv, self=[None]*len(m.dec), pos=0))

class Eng:
    def __init__(s, name, m): s.name, s.m = name, m.eval()
    def prefill(s, ids):
        if s.name == "B_encdec": return B_prefill(s.m, ids)
        f = A_forward if s.name == "A_decoder" else G_forward
        lg, layers = f(s.m, ids, None, 0); return lg, (layers, ids.shape[1])
    def step(s, tok, st):
        if s.name == "B_encdec": return B_step(s.m, tok, st)
        f = A_forward if s.name == "A_decoder" else G_forward
        lg, layers = f(s.m, tok[:, None], st[0], st[1]); return lg, (layers, st[1] + 1)

NEG = -float("inf")
@torch.no_grad()
def generate(eng, docs, bs=16, maxnew=SUM_TOK, min_new=30, rp=1.15, ng=3):
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]))
    out = [None]*len(docs); t0 = time.time()
    for bi in range(0, len(order), bs):
        idxs = order[bi:bi+bs]; n = len(idxs)
        L = min(len(docs[i]) for i in idxs)
        x = torch.tensor([ht(docs[i], L) for i in idxs], device=DEVICE)
        if eng.name != "B_encdec": x = torch.cat([x, torch.full((n, 1), BOS_ID, device=DEVICE)], 1)
        gen = [[] for _ in range(n)]; done = [False]*n; maps = [dict() for _ in range(n)]
        with torch.autocast("cuda", dtype=torch.float16):
            lg, st = eng.prefill(x)
            for s in range(maxnew):
                lg = lg.clone(); lg[:, [PAD_ID, UNK_ID, BOS_ID]] = NEG
                if s < min_new: lg[:, EOS_ID] = NEG
                for b in range(n):
                    if gen[b]:
                        ids = torch.tensor(list(set(gen[b][-200:])), device=DEVICE); v = lg[b, ids]
                        lg[b, ids] = torch.where(v > 0, v/rp, v*rp)
                    if ng > 1 and len(gen[b]) >= ng-1:
                        ban = maps[b].get(tuple(gen[b][-(ng-1):]))
                        if ban: lg[b, list(ban)] = NEG
                nxt = lg.argmax(-1); nl = nxt.tolist()
                for b in range(n):
                    if done[b]: continue
                    t = nl[b]
                    if t == EOS_ID: done[b] = True; continue
                    if ng > 1 and len(gen[b]) >= ng-1:
                        maps[b].setdefault(tuple(gen[b][-(ng-1):]), set()).add(t)
                    gen[b].append(t)
                if all(done): break
                lg, st = eng.step(nxt, st)
        for i, g in zip(idxs, gen):
            out[i] = tok.decode(g).strip() or "The court heard the matter and passed the order."
        print(f"  {min(bi+bs, len(order))}/{len(order)} | {time.time()-t0:.0f}s", flush=True)
    return out

In [6]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=16, alpha=32, drop=0.05):
        super().__init__(); self.base = base; self.scale = alpha / r; self.drop = nn.Dropout(drop)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class LoRMALinear(nn.Module):
    """Low-rank MULTIPLICATIVE adaptation: W' = (I + B A) W  ->  y = Wx + B(A(Wx))."""
    def __init__(self, base, r=16, alpha=32):
        super().__init__(); self.base = base; self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

class Adapter(nn.Module):
    def __init__(self, dim, r=64):
        super().__init__(); self.down = nn.Linear(dim, r); self.up = nn.Linear(r, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(F.gelu(self.down(x)))

def wrap_linears(model, wrapper, targets):
    jobs = [(p, n, c) for p in model.modules() for n, c in p.named_children()
            if isinstance(c, nn.Linear) and n in targets]
    for p, n, c in jobs: setattr(p, n, wrapper(c))

def add_adapters(model, pfeiffer=False, r=64, dim=768):
    # Houlsby: adapter after attention output AND after FFN. Pfeiffer: adapter after FFN only.
    for b in list(model.enc) + list(model.dec):
        if not pfeiffer:
            attns = [b.attn] if hasattr(b, "attn") else [b.self_attn, b.cross]
            for a in attns: a.proj = nn.Sequential(a.proj, Adapter(dim, r))
        b.mlp = nn.Sequential(b.mlp, Adapter(dim, r))

TARGETS = {"A_decoder": {"qkv","proj","w1","w2","w3"},
           "C_hybrid":  {"inp","out","q","k","v","proj","w1","w2","w3"},
           "D_efficient": {"q","k","v","proj","w1","w2","w3"}}
# experiment id -> (phase-1 model, method)
EXPS = {"E1": ("A_decoder", "lora"),
        "E2": ("B_encdec", "houlsby"),
        "E3": ("C_hybrid", "lorma"),
        "E4": ("D_efficient", "kd"),        # knowledge distillation, teacher = E1
        "E5": ("B_encdec", "pfeiffer"),
        "E6": ("D_efficient", "full")}      # plain full fine-tuning baseline (no teacher)
def tag_of(eid): return f"{eid}_{EXPS[eid][0]}_{EXPS[eid][1]}"

def make_model(name, meth, r=16):
    m = build(name); m.load_state_dict(torch.load(ckpt_path(name), map_location="cpu"))
    if meth not in ("kd", "full"):
        for p in m.parameters(): p.requires_grad = False
        if meth == "lora":     wrap_linears(m, lambda l: LoRALinear(l, r, 2*r), TARGETS[name])
        if meth == "lorma":    wrap_linears(m, lambda l: LoRMALinear(l, r, 2*r), TARGETS[name])
        if meth == "houlsby":  add_adapters(m, pfeiffer=False)
        if meth == "pfeiffer": add_adapters(m, pfeiffer=True)
    m = m.to(DEVICE)
    tp = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name} [{meth}] total {count_params(m)/1e6:.1f}M | trainable {tp/1e6:.2f}M")
    return m

def save_trained(tag, m):
    torch.save({k: v.detach().cpu() for k, v in m.named_parameters() if v.requires_grad}, f"{OUT}/{tag}.pt")
def load_trained(eid):
    name, meth = EXPS[eid]
    m = make_model(name, meth); m.load_state_dict(torch.load(f"{OUT}/{tag_of(eid)}.pt"), strict=False); return m

In [7]:
def make_batch(idxs):
    docs = [doc_ids[i] for i in idxs]; L = min(len(d) for d in docs); docs = [ht(d, L) for d in docs]
    sums = [sum_ids[i] for i in idxs]; S = max(len(s) for s in sums); n = len(idxs)
    Sp = torch.full((n, S), PAD_ID, dtype=torch.long); T = torch.full((n, S), -100, dtype=torch.long)
    for i, s in enumerate(sums):
        Sp[i, :len(s)] = torch.tensor(s); T[i, :len(s)] = torch.tensor(s)
    return torch.tensor(docs).to(DEVICE), Sp.to(DEVICE), T.to(DEVICE)

def logits_for(model, kind, D, Sp):
    n = D.size(0); bos = torch.full((n, 1), BOS_ID, device=DEVICE)
    if kind == "B_encdec":
        return model.decode(torch.cat([bos, Sp[:, :-1]], 1), model.encode(D))
    L = D.size(1)
    return model(torch.cat([D, bos, Sp], 1))[0][:, L:L + Sp.size(1)]

def train(name, model, steps, lr, tag, teacher=None, teacher_name=None, kd_alpha=0.5, Tmp=2.0, mbs=2, accum=8, save_every=200):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda"); it = batches(train_idx, mbs); model.train(); t0 = time.time()
    for step in range(steps):
        cur = lr * min(1, (step+1)/30) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * step / steps)))
        for g in opt.param_groups: g["lr"] = cur
        run = 0.0
        for _ in range(accum):
            D, Sp, T = make_batch(next(it))
            with torch.autocast("cuda", dtype=torch.float16):
                lg = logits_for(model, name, D, Sp)
                ce = F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), T.reshape(-1), ignore_index=-100)
                loss = ce
                if teacher is not None:
                    with torch.no_grad(): tl = logits_for(teacher, teacher_name, D, Sp).float()
                    mask = (T != -100).float()
                    kl = F.kl_div(F.log_softmax(lg.float()/Tmp, -1), F.softmax(tl/Tmp, -1), reduction="none").sum(-1)
                    kl = (kl * mask).sum() / mask.sum() * Tmp**2
                    loss = (1 - kd_alpha) * ce + kd_alpha * kl
            scaler.scale(loss / accum).backward(); run += ce.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step % 10 == 0: print(f"{name} step {step}/{steps} ce {run:.3f} lr {cur:.1e} | {(time.time()-t0)/60:.1f} min", flush=True)
        if (step+1) % save_every == 0: save_trained(tag, model)
    save_trained(tag, model); model.eval(); return model

from rouge_score import rouge_scorer
def evaluate(name, model, tag, n=200, **gen_kw):
    idxs = val_idx[:n]; eng = Eng(name, model)
    t0 = time.time(); preds = generate(eng, [doc_ids[i] for i in idxs], **gen_kw); secs = time.time() - t0
    refs = [sum_txt[i] for i in idxs]
    rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"])
    sc = [rs.score(r, p) for r, p in zip(refs, preds)]
    res = dict(exp=tag, params_M=round(count_params(model)/1e6, 1),
               trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, 2),
               rouge1=round(float(np.mean([s["rouge1"].fmeasure for s in sc])), 4),
               rougeL=round(float(np.mean([s["rougeL"].fmeasure for s in sc])), 4),
               avg_words=round(float(np.mean([len(p.split()) for p in preds])), 1),
               gen_sec_per_doc=round(secs/len(idxs), 2), peak_mem_GB=round(torch.cuda.max_memory_allocated()/1e9, 1))
    try:
        from bert_score import score as bs
        res["bertF1"] = round(bs(preds, refs, lang="en", device="cuda", batch_size=16)[2].mean().item(), 4)
    except Exception as e:
        res["bertF1"] = f"skipped ({type(e).__name__})"
    print(res); print("SAMPLE:", preds[0][:400]); return res

def predict_test(name, model, tag, **gen_kw):
    preds = generate(Eng(name, model), test_ids, **gen_kw)
    sub = [{"id": d["id"], "summary": p} for d, p in zip(te, preds)]
    assert len(sub) == len(te)
    d = f"{WORK}/submissions"; os.makedirs(d, exist_ok=True)
    jp = f"{d}/{tag}_prediction.json"; json.dump(sub, open(jp, "w"), ensure_ascii=False)
    zp = f"{d}/{tag}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")
    print("saved", zp); return sub

RESULTS = []
def run_experiment(eid, steps, lr, teacher=None, teacher_name=None):
    name, meth = EXPS[eid]; tag = tag_of(eid)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = make_model(name, meth)
    m = train(name, m, steps, lr, tag, teacher=teacher, teacher_name=teacher_name)
    RESULTS.append(evaluate(name, m, tag)); json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1)
    predict_test(name, m, tag); return m

## E1 - LoRA on decoder (also KD teacher). Smoke-test with steps=60 first; on OOM use mbs=1, accum=16 in train().

In [8]:
mA = run_experiment("E1", steps=600, lr=5e-4)

A_decoder [lora] total 112.0M | trainable 2.51M
A_decoder step 0/600 ce 3.698 lr 1.7e-05 | 0.0 min
A_decoder step 10/600 ce 3.731 lr 1.8e-04 | 0.3 min
A_decoder step 20/600 ce 3.544 lr 3.5e-04 | 0.6 min
A_decoder step 30/600 ce 3.521 lr 5.0e-04 | 0.9 min
A_decoder step 40/600 ce 3.523 lr 5.0e-04 | 1.2 min
A_decoder step 50/600 ce 3.515 lr 4.9e-04 | 1.6 min
A_decoder step 60/600 ce 3.294 lr 4.9e-04 | 1.9 min
A_decoder step 70/600 ce 3.541 lr 4.9e-04 | 2.2 min
A_decoder step 80/600 ce 3.414 lr 4.8e-04 | 2.6 min
A_decoder step 90/600 ce 3.365 lr 4.8e-04 | 2.9 min
A_decoder step 100/600 ce 3.415 lr 4.7e-04 | 3.2 min
A_decoder step 110/600 ce 3.372 lr 4.6e-04 | 3.5 min
A_decoder step 120/600 ce 3.223 lr 4.6e-04 | 3.9 min
A_decoder step 130/600 ce 3.245 lr 4.5e-04 | 4.2 min
A_decoder step 140/600 ce 3.257 lr 4.4e-04 | 4.5 min
A_decoder step 150/600 ce 3.252 lr 4.3e-04 | 4.9 min
A_decoder step 160/600 ce 3.714 lr 4.3e-04 | 5.2 min
A_decoder step 170/600 ce 3.285 lr 4.2e-04 | 5.5 min
A_decoder

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E1_A_decoder_lora', 'params_M': 112.0, 'trainable_M': 2.51, 'rouge1': 0.3054, 'rougeL': 0.1705, 'avg_words': 202.5, 'gen_sec_per_doc': 0.44, 'peak_mem_GB': 3.4, 'bertF1': 0.8313}
SAMPLE: The petitioners were charged with offences under sections 302 and 120B of the Code of Criminal Procedure for the offence of murder and murder. The petitioners were convicted under section 302 read with section 34 I.P.C. and sentenced to life imprisonment for one year. The petition was filed by the State of Bihar challenging the validity of the said order. The writ petitions were dismissed by the C
  16/100 | 6s
  32/100 | 14s
  48/100 | 21s
  64/100 | 28s
  80/100 | 35s
  96/100 | 41s
  100/100 | 47s
saved /kaggle/working/submissions/E1_A_decoder_lora.zip


## E2 - Houlsby adapters on enc-dec

In [9]:
mB = run_experiment("E2", steps=600, lr=5e-4); del mB; torch.cuda.empty_cache()

B_encdec [houlsby] total 126.7M | trainable 2.97M
B_encdec step 0/600 ce 4.452 lr 1.7e-05 | 0.0 min
B_encdec step 10/600 ce 4.475 lr 1.8e-04 | 0.2 min
B_encdec step 20/600 ce 4.183 lr 3.5e-04 | 0.3 min
B_encdec step 30/600 ce 4.473 lr 5.0e-04 | 0.5 min
B_encdec step 40/600 ce 4.129 lr 5.0e-04 | 0.6 min
B_encdec step 50/600 ce 4.113 lr 4.9e-04 | 0.8 min
B_encdec step 60/600 ce 4.141 lr 4.9e-04 | 0.9 min
B_encdec step 70/600 ce 4.041 lr 4.9e-04 | 1.1 min
B_encdec step 80/600 ce 3.919 lr 4.8e-04 | 1.2 min
B_encdec step 90/600 ce 4.151 lr 4.8e-04 | 1.4 min
B_encdec step 100/600 ce 4.142 lr 4.7e-04 | 1.5 min
B_encdec step 110/600 ce 3.972 lr 4.6e-04 | 1.7 min
B_encdec step 120/600 ce 3.935 lr 4.6e-04 | 1.8 min
B_encdec step 130/600 ce 4.222 lr 4.5e-04 | 2.0 min
B_encdec step 140/600 ce 4.127 lr 4.4e-04 | 2.1 min
B_encdec step 150/600 ce 4.251 lr 4.3e-04 | 2.3 min
B_encdec step 160/600 ce 3.953 lr 4.3e-04 | 2.4 min
B_encdec step 170/600 ce 3.982 lr 4.2e-04 | 2.6 min
B_encdec step 180/600 ce 

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E2_B_encdec_houlsby', 'params_M': 126.7, 'trainable_M': 2.97, 'rouge1': 0.2415, 'rougeL': 0.1461, 'avg_words': 157.7, 'gen_sec_per_doc': 0.26, 'peak_mem_GB': 2.1, 'bertF1': 0.8216}
SAMPLE: The appellant was a tenant of the State of Maharashtra. The respondent filed an application under section 5(1) of the Act for the purpose of making a notice under section 3(2)(a) of that Act. The appellant, who was a member of the District Magistrate, held that he had no right to pay the amount of Rs. 1,000/ and that he was not entitled to any compensation. The High Court dismissed the appeal. In 
  16/100 | 4s
  32/100 | 8s
  48/100 | 12s
  64/100 | 16s
  80/100 | 20s
  96/100 | 25s
  100/100 | 28s
saved /kaggle/working/submissions/E2_B_encdec_houlsby.zip


## E3 - LoRMA on hybrid (compulsory)

In [10]:
mC = run_experiment("E3", steps=600, lr=5e-4); del mC; torch.cuda.empty_cache()

C_hybrid [lorma] total 117.5M | trainable 3.24M
C_hybrid step 0/600 ce 3.677 lr 1.7e-05 | 0.1 min
C_hybrid step 10/600 ce 3.677 lr 1.8e-04 | 0.5 min
C_hybrid step 20/600 ce 3.701 lr 3.5e-04 | 1.0 min
C_hybrid step 30/600 ce 3.670 lr 5.0e-04 | 1.4 min
C_hybrid step 40/600 ce 3.504 lr 5.0e-04 | 1.8 min
C_hybrid step 50/600 ce 3.455 lr 4.9e-04 | 2.3 min
C_hybrid step 60/600 ce 3.377 lr 4.9e-04 | 2.7 min
C_hybrid step 70/600 ce 3.404 lr 4.9e-04 | 3.2 min
C_hybrid step 80/600 ce 3.706 lr 4.8e-04 | 3.6 min
C_hybrid step 90/600 ce 3.152 lr 4.8e-04 | 4.0 min
C_hybrid step 100/600 ce 3.552 lr 4.7e-04 | 4.5 min
C_hybrid step 110/600 ce 3.497 lr 4.6e-04 | 5.0 min
C_hybrid step 120/600 ce 3.457 lr 4.6e-04 | 5.4 min
C_hybrid step 130/600 ce 3.610 lr 4.5e-04 | 5.9 min
C_hybrid step 140/600 ce 3.457 lr 4.4e-04 | 6.3 min
C_hybrid step 150/600 ce 3.543 lr 4.3e-04 | 6.8 min
C_hybrid step 160/600 ce 3.421 lr 4.3e-04 | 7.2 min
C_hybrid step 170/600 ce 3.561 lr 4.2e-04 | 7.6 min
C_hybrid step 180/600 ce 3.

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E3_C_hybrid_lorma', 'params_M': 117.5, 'trainable_M': 3.24, 'rouge1': 0.2809, 'rougeL': 0.1599, 'avg_words': 175.1, 'gen_sec_per_doc': 0.43, 'peak_mem_GB': 4.0, 'bertF1': 0.8305}
SAMPLE: The petitioners were arrested on the same day and they were arrested by the police. The petitioners were tried for offences under Sections 420, 467 and 468 of I.P.C. and Section 3(1) (b) of the Prevention of Corruption Act, 1947. The allegations in the complaint were that the petitioners had been falsely implicated in a case registered against them under sections 147, 148, 149, 307, 504, 506, 427,
  16/100 | 7s
  32/100 | 13s
  48/100 | 20s
  64/100 | 27s
  80/100 | 33s
  96/100 | 40s
  100/100 | 46s
saved /kaggle/working/submissions/E3_C_hybrid_lorma.zip


## E4 - Knowledge distillation (teacher E1 A_decoder -> student D_efficient)

In [11]:
# if the session restarted: mA = load_trained("E1")
mA.eval()
for p in mA.parameters(): p.requires_grad = False
mD = run_experiment("E4", steps=600, lr=3e-4, teacher=mA, teacher_name="A_decoder"); del mD; torch.cuda.empty_cache()

D_efficient [kd] total 85.9M | trainable 85.94M
D_efficient step 0/600 ce 3.818 lr 1.0e-05 | 0.0 min
D_efficient step 10/600 ce 3.605 lr 1.1e-04 | 0.5 min
D_efficient step 20/600 ce 3.573 lr 2.1e-04 | 1.0 min
D_efficient step 30/600 ce 3.606 lr 3.0e-04 | 1.5 min
D_efficient step 40/600 ce 3.769 lr 3.0e-04 | 1.9 min
D_efficient step 50/600 ce 3.486 lr 3.0e-04 | 2.4 min
D_efficient step 60/600 ce 3.577 lr 2.9e-04 | 2.9 min
D_efficient step 70/600 ce 3.338 lr 2.9e-04 | 3.3 min
D_efficient step 80/600 ce 3.403 lr 2.9e-04 | 3.8 min
D_efficient step 90/600 ce 3.258 lr 2.9e-04 | 4.3 min
D_efficient step 100/600 ce 3.503 lr 2.8e-04 | 4.8 min
D_efficient step 110/600 ce 3.554 lr 2.8e-04 | 5.2 min
D_efficient step 120/600 ce 3.450 lr 2.7e-04 | 5.7 min
D_efficient step 130/600 ce 3.311 lr 2.7e-04 | 6.2 min
D_efficient step 140/600 ce 3.335 lr 2.7e-04 | 6.6 min
D_efficient step 150/600 ce 3.723 lr 2.6e-04 | 7.1 min
D_efficient step 160/600 ce 3.317 lr 2.6e-04 | 7.6 min
D_efficient step 170/600 ce 

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E4_D_efficient_kd', 'params_M': 85.9, 'trainable_M': 85.94, 'rouge1': 0.3192, 'rougeL': 0.1759, 'avg_words': 206.5, 'gen_sec_per_doc': 0.31, 'peak_mem_GB': 4.2, 'bertF1': 0.8349}
SAMPLE: The petitioners were charged with offences under sections 302 and 307 read with section 34 of the Indian Penal Code. The prosecution case was that on the basis of a complaint filed by the State, the complainant had been given an opportunity to show cause why they should not be released on bail. The complainant, who was also present at the time when the complaint was pending before the Court, conte
  16/100 | 5s
  32/100 | 10s
  48/100 | 15s
  64/100 | 20s
  80/100 | 25s
  96/100 | 30s
  100/100 | 33s
saved /kaggle/working/submissions/E4_D_efficient_kd.zip


## E5 - Pfeiffer adapters on enc-dec (adapter after FFN only)

In [12]:
mB2 = run_experiment("E5", steps=600, lr=5e-4); del mB2; torch.cuda.empty_cache()

B_encdec [pfeiffer] total 124.9M | trainable 1.19M
B_encdec step 0/600 ce 4.288 lr 1.7e-05 | 0.0 min
B_encdec step 10/600 ce 4.326 lr 1.8e-04 | 0.2 min
B_encdec step 20/600 ce 4.237 lr 3.5e-04 | 0.3 min
B_encdec step 30/600 ce 4.256 lr 5.0e-04 | 0.4 min
B_encdec step 40/600 ce 4.609 lr 5.0e-04 | 0.6 min
B_encdec step 50/600 ce 4.156 lr 4.9e-04 | 0.7 min
B_encdec step 60/600 ce 4.221 lr 4.9e-04 | 0.8 min
B_encdec step 70/600 ce 4.219 lr 4.9e-04 | 1.0 min
B_encdec step 80/600 ce 4.283 lr 4.8e-04 | 1.1 min
B_encdec step 90/600 ce 4.080 lr 4.8e-04 | 1.3 min
B_encdec step 100/600 ce 4.238 lr 4.7e-04 | 1.4 min
B_encdec step 110/600 ce 4.052 lr 4.6e-04 | 1.6 min
B_encdec step 120/600 ce 3.962 lr 4.6e-04 | 1.7 min
B_encdec step 130/600 ce 4.104 lr 4.5e-04 | 1.8 min
B_encdec step 140/600 ce 4.099 lr 4.4e-04 | 2.0 min
B_encdec step 150/600 ce 4.156 lr 4.3e-04 | 2.1 min
B_encdec step 160/600 ce 4.230 lr 4.3e-04 | 2.3 min
B_encdec step 170/600 ce 4.045 lr 4.2e-04 | 2.4 min
B_encdec step 180/600 ce

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E5_B_encdec_pfeiffer', 'params_M': 124.9, 'trainable_M': 1.19, 'rouge1': 0.2534, 'rougeL': 0.1524, 'avg_words': 177.5, 'gen_sec_per_doc': 0.24, 'peak_mem_GB': 2.0, 'bertF1': 0.8195}
SAMPLE: The appellant was a tenant of the deceased. The respondent filed an application under section 482 of the Code of Criminal Procedure for the offence punishable under section 302 read with section 34 of the Indian Penal Code, 1973. The appellant was convicted and sentenced to imprisonment for life. The trial court held that the appellant had been acquitted by the High Court and the High Courts below
  16/100 | 4s
  32/100 | 7s
  48/100 | 11s
  64/100 | 15s
  80/100 | 18s
  96/100 | 22s
  100/100 | 25s
saved /kaggle/working/submissions/E5_B_encdec_pfeiffer.zip


## E6 - Full fine-tuning of D_efficient, no teacher (baseline for the KD and PEFT comparisons)

In [13]:
mD2 = run_experiment("E6", steps=600, lr=1e-4); del mD2; torch.cuda.empty_cache()

D_efficient [full] total 85.9M | trainable 85.94M
D_efficient step 0/600 ce 3.669 lr 3.3e-06 | 0.0 min
D_efficient step 10/600 ce 3.447 lr 3.7e-05 | 0.4 min
D_efficient step 20/600 ce 3.726 lr 7.0e-05 | 0.7 min
D_efficient step 30/600 ce 3.689 lr 9.9e-05 | 1.0 min
D_efficient step 40/600 ce 3.164 lr 9.9e-05 | 1.4 min
D_efficient step 50/600 ce 3.384 lr 9.8e-05 | 1.7 min
D_efficient step 60/600 ce 3.497 lr 9.8e-05 | 2.0 min
D_efficient step 70/600 ce 3.352 lr 9.7e-05 | 2.4 min
D_efficient step 80/600 ce 3.487 lr 9.6e-05 | 2.7 min
D_efficient step 90/600 ce 3.295 lr 9.5e-05 | 3.0 min
D_efficient step 100/600 ce 3.444 lr 9.4e-05 | 3.4 min
D_efficient step 110/600 ce 3.356 lr 9.3e-05 | 3.7 min
D_efficient step 120/600 ce 3.207 lr 9.1e-05 | 4.0 min
D_efficient step 130/600 ce 3.248 lr 9.0e-05 | 4.4 min
D_efficient step 140/600 ce 3.399 lr 8.8e-05 | 4.7 min
D_efficient step 150/600 ce 3.143 lr 8.7e-05 | 5.0 min
D_efficient step 160/600 ce 3.039 lr 8.5e-05 | 5.4 min
D_efficient step 170/600 c

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E6_D_efficient_full', 'params_M': 85.9, 'trainable_M': 85.94, 'rouge1': 0.2918, 'rougeL': 0.1626, 'avg_words': 188.2, 'gen_sec_per_doc': 0.32, 'peak_mem_GB': 4.0, 'bertF1': 0.8297}
SAMPLE: The petitioners, who were members of the Central Bureau of Investigation and Police, were charged with offences under sections 302/149, 323/34 I.P.C., and section 5(1) (d) read with section 7/21 of the Prevention of Corruption Act, 1947. The prosecution case was that on the date of occurrence the petitioners were arrested and released on bail. The petitioners filed a writ petition in the High Cour
  16/100 | 5s
  32/100 | 10s
  48/100 | 15s
  64/100 | 20s
  80/100 | 25s
  96/100 | 30s
  100/100 | 34s
saved /kaggle/working/submissions/E6_D_efficient_full.zip


## Results table, generation-length study, and the 6 zips

In [14]:
import pandas as pd
print(pd.DataFrame(RESULTS).to_string(index=False))
for mn in [60, 100, 150]:
    print("min_new", mn, evaluate("A_decoder", mA, "E1_lenstudy", n=100, min_new=mn, maxnew=256))
zs = sorted(glob.glob(f"{WORK}/submissions/E*.zip"))
for z in zs: print(os.path.getsize(z)//1024, "KB", z)
print(len(zs), "zips ready")

                 exp  params_M  trainable_M  rouge1  rougeL  avg_words  gen_sec_per_doc  peak_mem_GB  bertF1
   E1_A_decoder_lora     112.0         2.51  0.3054  0.1705      202.5             0.44          3.4  0.8313
 E2_B_encdec_houlsby     126.7         2.97  0.2415  0.1461      157.7             0.26          2.1  0.8216
   E3_C_hybrid_lorma     117.5         3.24  0.2809  0.1599      175.1             0.43          4.0  0.8305
   E4_D_efficient_kd      85.9        85.94  0.3192  0.1759      206.5             0.31          4.2  0.8349
E5_B_encdec_pfeiffer     124.9         1.19  0.2534  0.1524      177.5             0.24          2.0  0.8195
 E6_D_efficient_full      85.9        85.94  0.2918  0.1626      188.2             0.32          4.0  0.8297
  16/100 | 7s
  32/100 | 14s
  48/100 | 21s
  64/100 | 28s
  80/100 | 36s
  96/100 | 43s
  100/100 | 49s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3083, 'rougeL': 0.1713, 'avg_words': 204.8, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
SAMPLE: The petitioners were charged with offences under sections 302 and 120B of the Code of Criminal Procedure for the offence of murder and murder. The petitioners were convicted under section 302 read with section 34 I.P.C. and sentenced to life imprisonment for one year. The petition was filed by the State of Bihar challenging the validity of the said order. The writ petitions were dismissed by the C
min_new 60 {'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3083, 'rougeL': 0.1713, 'avg_words': 204.8, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
  16/100 | 7s
  32/100 | 14s
  48/100 | 22s
  64/100 | 29s
  80/100 | 36s
  96/100 | 43s
  100/100 | 49s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3083, 'rougeL': 0.1713, 'avg_words': 204.8, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
SAMPLE: The petitioners were charged with offences under sections 302 and 120B of the Code of Criminal Procedure for the offence of murder and murder. The petitioners were convicted under section 302 read with section 34 I.P.C. and sentenced to life imprisonment for one year. The petition was filed by the State of Bihar challenging the validity of the said order. The writ petitions were dismissed by the C
min_new 100 {'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3083, 'rougeL': 0.1713, 'avg_words': 204.8, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
  16/100 | 7s
  32/100 | 14s
  48/100 | 21s
  64/100 | 28s
  80/100 | 35s
  96/100 | 43s
  100/100 | 49s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3088, 'rougeL': 0.1716, 'avg_words': 205.3, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
SAMPLE: The petitioners were charged with offences under sections 302 and 120B of the Code of Criminal Procedure for the offence of murder and murder. The petitioners were convicted under section 302 read with section 34 I.P.C. and sentenced to life imprisonment for one year. The petition was filed by the State of Bihar challenging the validity of the said order. The writ petitions were dismissed by the C
min_new 150 {'exp': 'E1_lenstudy', 'params_M': 112.0, 'trainable_M': 0.0, 'rouge1': 0.3088, 'rougeL': 0.1716, 'avg_words': 205.3, 'gen_sec_per_doc': 0.49, 'peak_mem_GB': 4.0, 'bertF1': 0.8313}
24 KB /kaggle/working/submissions/E1_A_decoder_lora.zip
10 KB /kaggle/working/submissions/E2_B_encdec_houlsby.zip
21 KB /kaggle/working/submissions/E3_C_hybrid_lorma.zip
27 KB /kaggle/working/submissions/E4_D_efficien